In [0]:
import os

dbutils.widgets.text("aws_access_key", "", "AWS Access Key")
dbutils.widgets.text("aws_secret_key", "", "AWS Secret Key")

aws_access_key = dbutils.widgets.get("aws_access_key")
aws_secret_key = dbutils.widgets.get("aws_secret_key")

os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key or os.getenv("AWS_ACCESS_KEY_ID", "")
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_key or os.getenv("AWS_SECRET_ACCESS_KEY", "")
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

In [0]:
df_application = (
    spark.read.format("csv")
    .option("header", "true")
    .option("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .option("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://meu-projeto-credito-raw/application_train.csv")
)

df_application.write.format("delta").mode("overwrite").saveAsTable("bronze_application")
print("Tabela bronze_application criada!")

Tabela bronze_application criada!


In [0]:
df_bureau = (
    spark.read.format("csv")
    .option("header", "true")
    .option("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .option("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://meu-projeto-credito-raw/bureau.csv")
)

df_bureau.write.format("delta").mode("overwrite").saveAsTable("bronze_bureau")
print("Tabela bronze_bureau criada!")

Tabela bronze_bureau criada!


In [0]:
df_previous_application = (
    spark.read.format("csv")
    .option("header", "true")
    .option("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .option("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://meu-projeto-credito-raw/previous_application.csv")
)

df_previous_application.write.format("delta").mode("overwrite").saveAsTable("bronze_previous_application")
print("Tabela bronze_previous_application criada!")
    

Tabela bronze_previous_application criada!


In [0]:
df_installments_payments = (
    spark.read.format("csv")
    .option("header", "true")
    .option("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .option("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://meu-projeto-credito-raw/installments_payments.csv")
)

df_installments_payments.write.format("delta").mode("overwrite").saveAsTable("bronze_installments_payments")
print("Tabela bronze_installments_payments criada!")
# Fim da camada Bronze

Tabela bronze_installments_payments criada!


In [0]:
# --- CAMADA SILVER: FEATURE ENGINEERING ---
from pyspark.sql import functions as F
# Inicio da camada Silver
application = spark.table("bronze_application")
bureau = spark.table("bronze_bureau")
previous = spark.table("bronze_previous_application")
installments = spark.table("bronze_installments_payments")

features_bureau = bureau.groupBy("SK_ID_CURR").agg(
    F.count("SK_ID_CURR").alias("qtd_loan_market"),
    F.sum(F.when(F.col("CREDIT_ACTIVE") == "Active", 1).otherwise(0)).alias("qtd_loans_active")
)

features_previous = previous.groupBy("SK_ID_CURR").agg(
    F.count("SK_ID_PREV").alias("qtd_previous_orders"),
    F.avg(F.col("AMT_ANNUITY").cast("double")).alias("previous_parcel_average_value"),
)

installments_with_delay = installments.withColumn(
    "days_delay",
    F.col("DAYS_ENTRY_PAYMENT").cast("double").cast("int") - F.col("DAYS_INSTALMENT").cast("double").cast("int")
)

features_installments = installments_with_delay.groupBy("SK_ID_CURR").agg(
    F.max(F.when(F.col("days_delay") > 0, F.col("days_delay")).otherwise(0)).alias("biggest_delay_days")
)

silver_credit_features = (
    application.select("SK_ID_CURR", "TARGET", "AMT_CREDIT", "NAME_CONTRACT_TYPE")
    .join(features_bureau, on="SK_ID_CURR", how="left")
    .join(features_previous, on="SK_ID_CURR", how="left")
    .join(features_installments, on="SK_ID_CURR", how="left")
    .fillna(0, subset=["qtd_loan_market", "qtd_loans_active", "qtd_previous_orders", "biggest_delay_days"])
)

silver_credit_features.write.format("delta").mode("overwrite").saveAsTable("silver_credit_features")
print("Tabela silver_credit_features criada com sucesso!")

display(spark.table("silver_credit_features"))




Tabela silver_credit_features criada com sucesso!


SK_ID_CURR,TARGET,AMT_CREDIT,NAME_CONTRACT_TYPE,qtd_loan_market,qtd_loans_active,qtd_previous_orders,previous_parcel_average_value,biggest_delay_days
100140,0,675000.0,Revolving loans,0,0,0,null,0
100248,0,808650.0,Cash loans,3,0,4,9435.87,0
101065,0,370629.0,Cash loans,2,2,2,10863.4275,5
101950,1,552555.0,Cash loans,5,3,3,17756.19,0
102226,0,585000.0,Cash loans,4,2,25,34633.93846153846,1
102394,0,62361.0,Cash loans,8,2,9,8346.223125,16
103827,0,112500.0,Cash loans,6,0,1,6748.38,0
103868,0,1335528.0,Cash loans,8,6,13,23189.733750000003,18
106608,0,1125000.0,Cash loans,0,0,2,36535.095,4
108037,0,188685.0,Cash loans,7,2,6,14330.5425,0
